# GDFM prototype notebook

This notebook is a guided tour of the current `GDFM` prototype.
It has two jobs:

1. check that the mechanics of the implementation are internally consistent
2. make the geometry of the model easier to see in a tiny 2D example

The order is intentional:

- first we set up the environment and import the prototype module
- then we instantiate a small model and run lightweight sanity checks
- then we verify that the new conversion rules are algebraically consistent
- finally we visualize a few explicit sampling steps in 2D


In [ ]:
# autoreload mode
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import torch.nn as nn

import numpy as np

import matplotlib.pyplot as plt

import seaborn as sns

sns.set_style("whitegrid")

In [ ]:
N = 1000
d = 2
x1 = torch.randn(N, d)
x2 = 5 + 0.2 * torch.randn(N, d)
x = torch.cat([x1, x2], dim=0)
print(x.shape)
plt.scatter(x[:, 0], x[:, 1])
plt.show()
x.shape

In [ ]:
import gdfm_components as g

## First tiny model instance

Here we build the smallest possible `GDFM` object for a 2D setting.
This is not meant for training quality yet.
It is just a convenient object to inspect so we can see the configuration,
network type, and path/parameterization choices in one place.


In [ ]:
# setup a config and make a gdfmodel from config
MAJ_CONFIG = g.GDFMConfig(family="fm", prediction_space="x", loss_space="x", data_dim=2)
model = g.GDFM(config=MAJ_CONFIG)
print(model)

In [ ]:
import torch
import matplotlib.pyplot as plt
from gdfm_components import GDFM, GDFMConfig

# ------------------------------------------------------------
# Make two FM models: uniform and logit-normal time sampling
# ------------------------------------------------------------

cfg_uniform = GDFMConfig(
    family="fm",
    prediction_space="x",
    loss_space="x",
    data_dim=8,
    fm_time_sampling="uniform",
    t_min=1e-3,
    t_max=1.0 - 1e-3,
)

cfg_logit = GDFMConfig(
    family="fm",
    prediction_space="x",
    loss_space="x",
    data_dim=8,
    fm_time_sampling="logit_normal",
    fm_logit_mu=0.0,
    fm_logit_sigma=1.0,
    t_min=1e-3,
    t_max=1.0 - 1e-3,
)

model_uniform = GDFM(cfg_uniform)
model_logit = GDFM(cfg_logit)

## Time-sampling intuition

Before doing model checks, it helps to inspect how time values are sampled.
This matters because the training loss is an expectation over `t`, so changing the sampling distribution changes which noise regimes the model sees most often.

The code below compares:

- uniform sampling over the allowed interval
- logit-normal sampling for flow matching
- several different `mu, sigma` settings for the logit-normal family

Pedagogically, this is the easiest way to build intuition for what the time-sampler hyperparameters are actually doing.


In [ ]:
n = 200_000
device = torch.device("cpu")

t_uniform = model_uniform.sample_t(n, device).cpu()
t_logit = model_logit.sample_t(n, device).cpu()

print("uniform")
print("  min :", t_uniform.min().item())
print("  max :", t_uniform.max().item())
print("  mean:", t_uniform.mean().item())
print("  std :", t_uniform.std().item())

print("\nlogit-normal")
print("  min :", t_logit.min().item())
print("  max :", t_logit.max().item())
print("  mean:", t_logit.mean().item())
print("  std :", t_logit.std().item())

# ------------------------------------------------------------
# Plot histograms
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))
plt.hist(t_uniform.numpy(), bins=100, alpha=0.6, label="uniform", density=True)
plt.hist(t_logit.numpy(), bins=100, alpha=0.6, label="logit-normal", density=True)
plt.xlabel("t")
plt.ylabel("density")
plt.title("Time sampling distributions")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

settings = [
    ("mu=0, sigma=0.5", 0.0, 0.5),
    ("mu=0, sigma=1.0", 0.0, 1.0),
    ("mu=0, sigma=2.0", 0.0, 2.0),
    ("mu=-2, sigma=1.0", -2.0, 1.0),
    ("mu=+2, sigma=1.0", 2.0, 1.0),
]

plt.figure(figsize=(10, 6))

for label, mu, sigma in settings:
    cfg = GDFMConfig(
        family="fm",
        prediction_space="x",
        loss_space="x",
        data_dim=8,
        fm_time_sampling="logit_normal",
        fm_logit_mu=mu,
        fm_logit_sigma=sigma,
        t_min=1e-3,
        t_max=1.0 - 1e-3,
    )
    model = GDFM(cfg)
    t = model.sample_t(n, device).cpu()
    plt.hist(t.numpy(), bins=100, density=True, alpha=0.25, label=label)

plt.xlabel("t")
plt.ylabel("density")
plt.title("FM logit-normal time sampling for different parameters")
plt.legend()
plt.show()

## Sanity-check model used below

The next block creates a slightly larger MLP-backed `GDFM` instance.
This one is still tiny, but large enough to run shape checks, a full loss pass,
and parameter counting without the outputs being too trivial.

What to notice:

- `family="fm"` chooses the interpolation path
- `prediction_space` and `loss_space` are independent choices
- `data_dim=32` means the network sees 32-dimensional flat vectors


In [ ]:
# create model

import torch
from gdfm_components import GDFM, GDFMConfig

device = torch.device("cpu")

# small sanity config
cfg = GDFMConfig(
    family="fm",
    prediction_space="x",
    loss_space="x",
    data_dim=32,
    hidden_dim=64,
    n_layers=3,
)

model = GDFM(cfg).to(device)
print(model)
print("---" * 90)
print(model.net)

## Shape check

This cell answers a very basic but important question:

> if we create a noisy point `z_t` and send it through the network,
> does the output have the same batch and feature shape?

That should be true for all parameterizations here, because the network always maps
`(z_t, t)` back into an object of the same ambient dimension.


In [ ]:
# ------------------------------------------------------------
# 1. shape check: input -> output
# ------------------------------------------------------------
B, D = 16, cfg.data_dim  # batch size and data dimension
x = torch.randn(B, D, device=device)
t = model.sample_t(B, device)
z = model.z_t_interpolate(x, torch.randn_like(x), t)

y = model.predict(z, t)

print("Input shape  (z):", z.shape)
print("Output shape (y):", y.shape)

assert y.shape == z.shape, "Mismatch: network does not preserve dimension"


## Loss and gradient flow check

Now we run one full call to `compute_loss(...)`.
Pedagogically, this is the first cell that exercises the whole training-side pipeline:

- sample noise and time
- build the interpolation `z_t`
- predict in the chosen prediction space
- convert into the chosen loss space if needed
- compare against the correct target

The backward pass is just as important as the scalar loss.
A finite loss is not enough if gradients do not actually reach the parameters.


In [ ]:
# ------------------------------------------------------------
# 2. full loss pass sanity
# ------------------------------------------------------------
loss = model.compute_loss(x)
print("Loss:", loss.item())

loss.backward()

has_grad = any(
    (p.grad is not None) and (p.grad.abs().sum() > 0) for p in model.parameters()
)

print("Gradients flowing:", has_grad)
assert has_grad, "No gradients flowing"

## Parameter count

This is a quick capacity check.
For prototype work, it helps to know whether a strange behavior is likely coming from:

- the math / conversions
- the solver
- or just a network that is much larger or smaller than intended


In [ ]:
# ------------------------------------------------------------
# 3. parameter count
# ------------------------------------------------------------
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters     : {n_params:,}")
print(f"Trainable parameters : {n_trainable:,}")

## Raw network printout

This tiny cell is just the most literal view of the neural network object.
It is useful as a quick check when you want to confirm the layer stack without the extra formatting of the summary helpers below.


In [ ]:
print(model.net)

## Readable model summary

The next helper cell prints the model in a more teaching-friendly way.
Instead of only showing the raw module tree, it also lists:

- the high-level `GDFM` configuration
- total and trainable parameter counts
- the shape flow through the linear layers

This is useful when you want to connect the abstract equations to the actual tensor shapes.


In [ ]:
## Better print model summary


def print_model_summary(model):
    print("=" * 60)
    print("MODEL SUMMARY")
    print("=" * 60)

    print(model)  # your custom __repr__

    print("\n--- Network ---")
    print(model.net)

    print("\n--- Parameter counts ---")
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total     : {total:,}")
    print(f"Trainable : {trainable:,}")

    print("\n--- Layer-wise params ---")
    for name, p in model.named_parameters():
        print(f"{name:40s} {tuple(p.shape)}  ({p.numel():,})")


def print_forward_shapes(model, x, t):
    print("=" * 60)
    print("FORWARD SHAPES")
    print("=" * 60)

    def hook(name):
        def fn(module, inp, out):
            in_shape = tuple(inp[0].shape) if isinstance(inp, tuple) else "?"
            out_shape = tuple(out.shape)
            print(f"{name:30s} {in_shape} -> {out_shape}")

        return fn

    handles = []
    for name, module in model.net.named_modules():
        if isinstance(module, nn.Linear):
            handles.append(module.register_forward_hook(hook(name)))

    z = x
    _ = model.net(z, t)

    for h in handles:
        h.remove()


print_model_summary(model)
print("-" * 60)
print_forward_shapes(model, x, t)


## Conversion Cycle Checks

This section is about the new algebraic heart of the prototype: `convert(...)`.

The core idea is that once we know the interpolation

- `z_t = a(t) x + b(t) eps`

and the velocity definition

- `v_t = a'(t) x + b'(t) eps`

we should be able to move consistently between the three spaces:

- `x`
- `eps`
- `v`

The code cell below tests two kinds of consistency:

1. **Direct conversion checks**
   Starting from the true target in one space, convert it once and compare against the true target in the destination space.

2. **Cycle / round-trip checks**
   Convert to another space and then back again, for example `x -> eps -> x`.
   If the implementation is correct, we should recover the original quantity up to tiny numerical error.

Why this matters:
If these checks fail, then training in one representation and measuring loss in another would be mathematically inconsistent.


In [ ]:
torch.manual_seed(7)


def max_abs_err(a, b):
    return float((a - b).abs().max().item())


def run_conversion_cycle_checks(family, batch_size=2048):
    cfg = g.GDFMConfig(
        family=family,
        prediction_space="v",
        loss_space="v",
        data_dim=2,
        t_min=1e-3,
        t_max=1.0 - 1e-3,
        fm_time_sampling="logit_normal" if family == "fm" else "uniform",
    )
    model = g.GDFM(cfg)

    x = torch.randn(batch_size, cfg.data_dim)
    eps = torch.randn_like(x)
    t = model.sample_t(batch_size, x.device)
    z_t = model.z_t_interpolate(x, eps, t)

    truth = {
        space: model.get_true_target(x, eps, t, space) for space in ("x", "eps", "v")
    }

    direct_pairs = [
        ("x", "eps"),
        ("x", "v"),
        ("eps", "x"),
        ("eps", "v"),
        ("v", "x"),
        ("v", "eps"),
    ]
    cycle_pairs = [
        ("x", "eps", "x"),
        ("x", "v", "x"),
        ("eps", "x", "eps"),
        ("eps", "v", "eps"),
        ("v", "x", "v"),
        ("v", "eps", "v"),
    ]

    direct_rows = []
    for source, target in direct_pairs:
        converted = model.convert(truth[source], z_t, t, source=source, target=target)
        direct_rows.append((source, target, max_abs_err(converted, truth[target])))

    cycle_rows = []
    for source, mid, target in cycle_pairs:
        mid_value = model.convert(truth[source], z_t, t, source=source, target=mid)
        cycled = model.convert(mid_value, z_t, t, source=mid, target=target)
        cycle_rows.append((source, mid, target, max_abs_err(cycled, truth[target])))

    return direct_rows, cycle_rows


for family in ("fm", "vp"):
    direct_rows, cycle_rows = run_conversion_cycle_checks(family)

    print(f"\n=== {family.upper()} direct conversion checks ===")
    for source, target, err in direct_rows:
        print(f"{source:>3} -> {target:<3} | max abs err = {err:.3e}")

    print(f"\n=== {family.upper()} round-trip checks ===")
    for source, mid, target, err in cycle_rows:
        print(f"{source:>3} -> {mid:>3} -> {target:<3} | max abs err = {err:.3e}")

    worst_err = max(
        max(err for _, _, err in direct_rows),
        max(err for _, _, _, err in cycle_rows),
    )
    print(f"\nWorst error for {family.upper()}: {worst_err:.3e}")
    assert worst_err < 1e-4, (
        f"{family} conversion cycle check failed with error {worst_err:.3e}"
    )

print("\nAll conversion-cycle checks passed.")


## 2D Sampling Walkthrough

This section is not trying to show a well-trained generative model.
Instead, it isolates the **sampling mechanics** in the simplest visual setting.

We use a hand-crafted 2D velocity field so we can answer questions like:

- does the sampler move points in the expected direction?
- what does one Euler step actually do geometrically?
- how does Heun differ from Euler over a few steps?

How to read the plots:

- pale background arrows: the velocity field at a representative time
- `x` markers: particle locations at each solver step
- colored arrows: the update from one step to the next
- numbered labels: the time ordering along the first particle's path
- red star: the attractor / target point of the demo field

So this section is really a visual debugging aid for the ODE integrator.


### Sampling helper objects

The next code cell defines three ingredients used by the visualization:

- `DemoVelocityField`: a simple analytic 2D field with attraction plus a little swirl
- `rollout_sampling_steps(...)`: explicitly records each intermediate solver state
- `plot_sampling_rollout(...)`: turns those intermediate states into a readable plot

This is pedagogically useful because the normal `sample(...)` interface only returns the final sample.
For understanding the solver, we want the whole trajectory.


In [ ]:
class DemoVelocityField(nn.Module):
    """Small hand-crafted velocity field for visualizing the sampler in 2D."""

    def __init__(self, target=(1.35, 0.8), radial_gain=0.85, swirl_gain=0.35):
        super().__init__()
        self.target = nn.Parameter(
            torch.tensor(target, dtype=torch.float32), requires_grad=False
        )
        self.radial_gain = radial_gain
        self.swirl_gain = swirl_gain

    def forward(self, z, t):
        if t.dim() == 1:
            t = t.view(-1, 1)

        target = self.target.view(1, 2).to(z)
        radial = target - z
        swirl = torch.stack(
            [-(z[:, 1] - target[:, 1]), z[:, 0] - target[:, 0]],
            dim=1,
        )

        # More swirl early, more direct attraction late.
        time_scale = 0.35 + 0.65 * (1.0 - t)
        return self.radial_gain * radial + self.swirl_gain * time_scale * swirl


@torch.no_grad()
def rollout_sampling_steps(model, z0, n_steps=5, method="euler"):
    ts = torch.linspace(
        model.config.t_min, model.config.t_max, n_steps + 1, device=z0.device
    )
    states = [z0.clone()]
    z = z0.clone()
    batch_size = z.shape[0]

    for k in range(len(ts) - 1):
        dt = ts[k + 1] - ts[k]
        if method == "euler":
            v = model._velocity(z, ts[k].item(), batch_size)
            z = z + dt * v
        elif method == "heun":
            v1 = model._velocity(z, ts[k].item(), batch_size)
            z_tilde = z + dt * v1
            v2 = model._velocity(z_tilde, ts[k + 1].item(), batch_size)
            z = z + 0.5 * dt * (v1 + v2)
        else:
            raise ValueError(f"Unknown method: {method}")
        states.append(z.clone())

    return ts.cpu(), torch.stack(states).cpu()


@torch.no_grad()
def plot_sampling_rollout(ax, model, ts, states, title):
    xlim = (-2.8, 2.5)
    ylim = (-2.4, 2.6)

    gx, gy = np.meshgrid(np.linspace(*xlim, 16), np.linspace(*ylim, 16))
    grid = torch.tensor(np.stack([gx.ravel(), gy.ravel()], axis=1), dtype=torch.float32)
    t_ref = float(ts[min(2, len(ts) - 1)])
    vel = model._velocity(grid, t_ref, grid.shape[0]).cpu().numpy()

    ax.quiver(
        gx,
        gy,
        vel[:, 0].reshape(gx.shape),
        vel[:, 1].reshape(gy.shape),
        color="#cbd5e1",
        angles="xy",
        scale_units="xy",
        scale=7.5,
        width=0.003,
        alpha=0.95,
    )

    step_colors = plt.cm.viridis(np.linspace(0.12, 0.95, states.shape[0] - 1))
    states_np = states.numpy()

    for step in range(states.shape[0]):
        color = "#111827" if step == 0 else step_colors[step - 1]
        ax.scatter(
            states_np[step, :, 0],
            states_np[step, :, 1],
            marker="x",
            s=70,
            linewidths=2.0,
            color=color,
            zorder=3,
        )

    for step in range(states.shape[0] - 1):
        start = states_np[step]
        delta = states_np[step + 1] - states_np[step]
        ax.quiver(
            start[:, 0],
            start[:, 1],
            delta[:, 0],
            delta[:, 1],
            angles="xy",
            scale_units="xy",
            scale=1,
            color=step_colors[step],
            width=0.006,
            alpha=0.95,
            zorder=2,
        )

    first_path = states_np[:, 0, :]
    for step, point in enumerate(first_path):
        ax.text(
            point[0] + 0.05, point[1] + 0.05, str(step), fontsize=9, color="#111827"
        )

    target = model.net.target.detach().cpu().numpy()
    ax.scatter(
        [target[0]],
        [target[1]],
        marker="*",
        s=220,
        color="#ef4444",
        edgecolor="black",
        linewidth=0.8,
        zorder=4,
    )

    ax.set_title(title)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    ax.set_xlabel("z[0]")
    ax.set_ylabel("z[1]")
    ax.grid(alpha=0.25)


sample_cfg = g.GDFMConfig(
    family="fm",
    prediction_space="v",
    loss_space="v",
    data_dim=2,
    t_min=0.05,
    t_max=0.95,
)
sampling_demo = g.GDFM(sample_cfg)
sampling_demo.net = DemoVelocityField(
    target=(1.35, 0.8), radial_gain=0.85, swirl_gain=0.35
)


### Euler versus Heun on the same starting points

Here we start several particles from fixed 2D positions and run both solvers for the same number of steps.

What to look for:

- Euler usually follows the field more crudely and can overshoot more
- Heun uses a predictor-corrector update, so it often bends more smoothly
- both methods should move particles generally toward the red star in this demo

The printed coordinates underneath are there to complement the picture with exact numbers for one trajectory.


In [ ]:
z0 = torch.tensor(
    [
        [-2.2, -1.4],
        [-1.6, 1.5],
        [-0.8, -2.0],
        [0.5, -1.8],
        [1.9, -1.2],
        [-2.4, 0.2],
    ],
    dtype=torch.float32,
)

ts_euler, states_euler = rollout_sampling_steps(
    sampling_demo, z0, n_steps=5, method="euler"
)
ts_heun, states_heun = rollout_sampling_steps(
    sampling_demo, z0, n_steps=5, method="heun"
)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.6), constrained_layout=True)
plot_sampling_rollout(
    axes[0], sampling_demo, ts_euler, states_euler, "Euler rollout (5 steps)"
)
plot_sampling_rollout(
    axes[1], sampling_demo, ts_heun, states_heun, "Heun rollout (5 steps)"
)
plt.show()

print("First trajectory coordinates (Euler):")
for step, (t_val, point) in enumerate(
    zip(ts_euler.tolist(), states_euler[:, 0, :].tolist())
):
    print(f"  step {step}: t={t_val:.2f}, z=({point[0]: .3f}, {point[1]: .3f})")

print("\nFirst trajectory coordinates (Heun):")
for step, (t_val, point) in enumerate(
    zip(ts_heun.tolist(), states_heun[:, 0, :].tolist())
):
    print(f"  step {step}: t={t_val:.2f}, z=({point[0]: .3f}, {point[1]: .3f})")


## Internal sampler checks

This final section tests the model's **own sampling code paths** directly.
The goal here is narrower than the earlier visualization section:
we want to verify that the public sampler and the internal integrators agree.

Concretely, we will test:

- `model._velocity(...)`
- `model._sample_euler(...)`
- `model._sample_heun(...)`
- `model.sample(...)`

The key idea is to fix the random seed, reconstruct the exact initial noise used by `sample(...)`,
and then check that the internal integrator returns the same final tensor.


### Trace the internal integrators step by step

`_sample_euler(...)` and `_sample_heun(...)` only return the final sample.
That is good for normal use, but not for understanding the solver.

So in the next cell we create a tiny tracing helper that uses the **same update equations** as the internal methods,
but stores every intermediate state. This lets us visualize the actual path taken by the internal integrator.


In [ ]:
@torch.no_grad()
def trace_internal_integrator(model, z0, n_steps=6, method="euler"):
    ts = torch.linspace(
        model.config.t_min, model.config.t_max, n_steps + 1, device=z0.device
    )
    states = [z0.clone()]
    z = z0.clone()
    batch_size = z.shape[0]

    if method == "euler":
        for k in range(len(ts) - 1):
            dt = ts[k + 1] - ts[k]
            v = model._velocity(z, ts[k].item(), batch_size)
            z = z + dt * v
            states.append(z.clone())
    elif method == "heun":
        for k in range(len(ts) - 1):
            dt = ts[k + 1] - ts[k]
            v1 = model._velocity(z, ts[k].item(), batch_size)
            z_tilde = z + dt * v1
            v2 = model._velocity(z_tilde, ts[k + 1].item(), batch_size)
            z = z + 0.5 * dt * (v1 + v2)
            states.append(z.clone())
    else:
        raise ValueError(f"Unknown method: {method}")

    return ts.cpu(), torch.stack(states).cpu()


torch.manual_seed(999)
z0_internal = torch.randn(8, 2)

ts_internal_euler, states_internal_euler = trace_internal_integrator(
    sampling_demo, z0_internal, n_steps=6, method="euler"
)
ts_internal_heun, states_internal_heun = trace_internal_integrator(
    sampling_demo, z0_internal, n_steps=6, method="heun"
)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.6), constrained_layout=True)
plot_sampling_rollout(
    axes[0],
    sampling_demo,
    ts_internal_euler,
    states_internal_euler,
    "Internal _sample_euler traced",
)
plot_sampling_rollout(
    axes[1],
    sampling_demo,
    ts_internal_heun,
    states_internal_heun,
    "Internal _sample_heun traced",
)
plt.show()

print("Max final-state difference between traced Euler and traced Heun:")
print(float((states_internal_euler[-1] - states_internal_heun[-1]).abs().max().item()))


### Public `sample(...)` output clouds

Finally, we call the model's public sampler exactly the way a normal user would:
just a shape, a step count, and a method.

This plot is useful for seeing the **end result** of the internal sampler on a larger batch.
It complements the earlier step-by-step plots by showing the final sample clouds produced by
Euler and Heun under the same random seed.


In [ ]:
torch.manual_seed(2026)
public_euler = sampling_demo.sample((300, 2), n_steps=30, method="euler", device="cpu")

torch.manual_seed(2026)
public_heun = sampling_demo.sample((300, 2), n_steps=30, method="heun", device="cpu")

target = sampling_demo.net.target.detach().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12.8, 5.4), constrained_layout=True)
for ax, pts, title, color in [
    (axes[0], public_euler, "Public sample(...), Euler", "#2563eb"),
    (axes[1], public_heun, "Public sample(...), Heun", "#059669"),
]:
    arr = pts.cpu().numpy()
    ax.scatter(arr[:, 0], arr[:, 1], s=18, alpha=0.5, color=color, edgecolors="none")
    ax.scatter(
        [target[0]],
        [target[1]],
        marker="*",
        s=220,
        color="#ef4444",
        edgecolor="black",
        linewidth=0.8,
    )
    ax.set_title(title)
    ax.set_xlabel("z[0]")
    ax.set_ylabel("z[1]")
    ax.set_aspect("equal")
    ax.grid(alpha=0.25)

plt.show()

print("Mean final location (Euler):", public_euler.mean(dim=0))
print("Mean final location (Heun): ", public_heun.mean(dim=0))
print(
    "Max |Euler - Heun| over paired final samples:",
    float((public_euler - public_heun).abs().max().item()),
)


In [ ]:
# lets train a fucking model now:

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print("-" * 80)
########################################
# set up dataset
N = 4
DIM = 2
D = torch.randn(N, DIM)
# unit normalize each row of D
D = D / torch.linalg.norm(D, dim=1, keepdim=True)
plt.scatter(D[:, 0], D[:, 1], alpha=0.5, s=4)
plt.show()
print("scatter plot")
print("-" * 80)


# make
model = GDFM(
    GDFMConfig(family="fm", prediction_space="x", loss_space="x", data_dim=DIM)
).to(DEVICE)
print(f"make model: {model}")
print(f"Better print: {g.print_model_summary(model)}")

# set optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# set up training loop
for i in range(10000):
    optimizer.zero_grad()
    loss = model.compute_loss(D)
    loss.backward()
    optimizer.step()
    if i % 100 == 0:
        print(f"loss: {loss.item()}")


In [ ]:
model.eval()

# sample a bunch of points from the model
with torch.no_grad():
    z = torch.randn(100, 2)
    x = model.sample(z=z, n_steps=100, method="heun")
    print(x.shape)

plt.scatter(z[:, 0], z[:, 1], alpha=0.1, label="noise")
plt.scatter(D[:, 0], D[:, 1], alpha=0.5, label="data")
plt.scatter(x[:, 0], x[:, 1], alpha=0.5, label="samples")
plt.legend()
plt.show()

In [ ]:
# def trace_internal_integrator(model, z0, n_steps=6, method="euler"):

trace_internal_integrator(model, z0, n_steps=6, method="euler")


torch.manual_seed(999)


ts_internal_euler, states_internal_euler = trace_internal_integrator(
    model=model, z0=z, n_steps=6, method="euler"
)
ts_internal_heun, states_internal_heun = trace_internal_integrator(
    model=model, z0=z, n_steps=6, method="heun"
)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.6), constrained_layout=True)
plot_sampling_rollout(
    axes[0],
    model,
    ts_internal_euler,
    states_internal_euler,
    "Internal _sample_euler traced",
)
plot_sampling_rollout(
    axes[1],
    model,
    ts_internal_heun,
    states_internal_heun,
    "Internal _sample_heun traced",
)
plt.show()


In [ ]:
# %% GDFM 2D Sanity Check — All 9 Cells
import math
import torch
import numpy as np
import matplotlib.pyplot as plt
from gdfm_components import GDFM, GDFMConfig

# ---- config ----
FAMILY = "fm"  # "fm" or "vp"
DATASET = "ring"  # "ring", "moons", "gaussians"
N_DATA = 200
N_STEPS = 50000
LR = 1e-3
BATCH_SIZE = 256
NOISE = 0.000
N_SAMPLE = 2000
N_ODE_STEPS = 500


# ---- datasets ----
def sample_ring(n, noise=0.05):
    theta = torch.rand(n) * 2 * math.pi
    x = torch.stack([theta.cos(), theta.sin()], dim=1)
    return x + noise * torch.randn_like(x)


def sample_moons(n, noise=0.05):
    from sklearn.datasets import make_moons

    data, _ = make_moons(n_samples=n, noise=noise)
    return torch.tensor(data, dtype=torch.float32)


def sample_gaussians(n, n_clusters=8, std=0.05):
    angles = torch.linspace(0, 2 * math.pi, n_clusters + 1)[:-1]
    centers = torch.stack([angles.cos(), angles.sin()], dim=1) * 2.0
    idx = torch.randint(0, n_clusters, (n,))
    return centers[idx] + std * torch.randn(n, 2)


datasets = {"ring": sample_ring, "moons": sample_moons, "gaussians": sample_gaussians}
data = datasets[DATASET](N_DATA, noise=NOISE)
data_np = data.numpy()

# ---- shared initial noise for fair comparison ----
z_init = torch.randn(N_SAMPLE, 2)

# ---- train + sample all 9 cells ----
print("-" * 80)

cfg = GDFMConfig(
    family=FAMILY,
    prediction_space=pred,
    loss_space=loss_s,
    data_dim=2,
    hidden_dim=128,
    n_layers=3,
    time_emb_dim=32,
)
model = GDFM(cfg)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
print(f"model: {g.print_model_summary(model)}")
print("-" * 80)


for step in range(N_STEPS):
    idx = torch.randint(0, N_DATA, (BATCH_SIZE,))
    loss = model.compute_loss(data[idx])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # lets print shit
    if step % (N_STEPS // 10) == 0:
        print(f"step {step} loss={loss.item():.4f}")

    if step % (N_STEPS // 4) == 0:
        # plot
        model.eval()
        z0 = torch.randn(N_SAMPLE, 2)
        samples = model.sample(z0, n_steps=N_ODE_STEPS, method="heun").numpy()
        ax.scatter(samples[:, 0], samples[:, 1], s=1, alpha=0.3, c="tab:blue")
        ax.scatter(data_np[:, 0], data_np[:, 1], s=1, alpha=0.3, c="gray")
        ax.scatter(z_init[:, 0], z_init[:, 1], s=1, alpha=0.1, c="tab:red")
        model.train()


In [ ]:
# for i, pred in enumerate(spaces):
#    for j, loss_s in enumerate(spaces):

spaces = ["x", "eps", "v"]
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
fig.suptitle(
    f"3×3 grid | {FAMILY} | {DATASET} | n={N_DATA} | {N_STEPS} steps", fontsize=13
)


ax = axes[i, j]
print(f"pred={pred}, loss={loss_s}", end="  ", flush=True)
pred = "eps"
loss_s = "eps"


print(f"loss={loss.item():.4f}")

# ---- config ----


N_ODE_STEPS = 10

samples = model.sample(z_init.clone(), n_steps=N_ODE_STEPS, method="euler").numpy()

ax.scatter(data_np[:, 0], data_np[:, 1], s=1, alpha=0.3, c="gray", label="data")
ax.scatter(
    samples[:, 0], samples[:, 1], s=1, alpha=0.3, c="tab:blue", label="generated"
)
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_aspect("equal")

if i == 0:
    ax.set_title(f"loss={loss_s}", fontsize=11, fontweight="bold")
if j == 0:
    ax.set_ylabel(f"pred={pred}", fontsize=11, fontweight="bold")

fig.tight_layout()
plt.show()

In [ ]:
print_model_summary(model)